# Assignment 10 - Problem 3
## Streaming vs. complete responses with the OpenAI API

**Name:** Hussein Jawad

In this problem I write two functions that take the same prompt and the same model settings:
1. `get_complete_response()` waits for the whole answer and returns it.
2. `get_streamed_response()` uses `stream=True` and prints the text while it arrives. It handles the `response.output_text.delta` events, rebuilds the full text without duplicating anything, and checks that the response finished successfully.

Steps:
1. Setup
2. The prompt and the settings (shared by both functions)
3. Function 1: complete response
4. Function 2: streaming response
5. Run both functions
6. Check the streamed text
7. What happens when a response doesn't finish

At the end are my observations and an appendix with the concepts.

## Step 1 - Setup

Same setup as Problem 1: install the libraries, mount my Google Drive folder (where the `.env` file with my API key is), and create the client.

In [1]:
# Install the OpenAI Python library and python-dotenv (reads the .env file)
%pip install -q --upgrade openai python-dotenv

In [2]:
import os   # file and folder functions

PROJECT_DIR = "/content/drive/MyDrive/CSCI-E-89-1-Deep-Learning/HW10"   # Drive folder with this notebook and .env

try:                                 # works only in Colab
    from google.colab import drive   # this import only works inside Colab
    drive.mount("/content/drive")    # connect my Google Drive to the Colab machine
    os.chdir(PROJECT_DIR)            # work inside the project folder so .env is found
except ImportError:                  # not in Colab
    pass                             # not in Colab: stay in the current folder

print("Working folder:", os.getcwd())               # show the folder we are working in
print(".env found    :", os.path.exists(".env"))    # check that the key file is there

Mounted at /content/drive
Working folder: /content/drive/MyDrive/CSCI-E-89-1-Deep-Learning/HW10
.env found    : True


In [3]:
import openai                                   # the OpenAI library (used here to print its version)
from dotenv import load_dotenv, find_dotenv     # read variables from the .env file
from openai import OpenAI                       # the client class used for all API calls

load_dotenv(find_dotenv(usecwd=True), override=True)   # load OPENAI_API_KEY from .env

MODEL = "gpt-5.4"   # same model as in Problem 1
client = OpenAI()   # create the client; it reads OPENAI_API_KEY from the environment

key = os.getenv("OPENAI_API_KEY")                                 # get the key that was loaded
assert key, "OPENAI_API_KEY not found - check the .env file"      # stop here if the key is missing
print("openai version:", openai.__version__)                      # library version
print("Key           :", key[:3] + "..." + key[-4:])              # masked key, never the full key

openai version: 3.54.0
Key           : sk-...iIoA


## Step 2 - The prompt and the settings

Both functions get exactly the same prompt and the same settings, so the only difference between them is streaming. Like in Problem 1, I use temperature 0 with no reasoning, and allow up to 800 output tokens.

In [4]:
PROMPT = ("Explain in about 150 words what it means to stream a response from a large language model, "   # the prompt text, split over two lines
          "and why it makes a chat application feel faster.")    # the same prompt for both functions

SETTINGS = {                                  # the same model settings for both functions
    "model": MODEL,                           # gpt-5.4
    "reasoning": {"effort": "none"},          # no reasoning, so temperature can be set
    "temperature": 0,                         # least random output
    "max_output_tokens": 800,                 # upper limit on the answer length
}

## Step 3 - Function 1: complete response

`get_complete_response()` makes one normal call. Nothing comes back until the model has written the whole answer. It returns the text, the response object and how long the call took. It also checks the status: if it isn't `completed` (for example `incomplete` when the answer was cut off), it raises an error.

In [5]:
import time   # to measure how long each call takes

def get_complete_response(prompt, **settings):   # function 1
    # One normal call: wait for the whole answer, then return it
    start = time.time()                                              # start the timer
    response = client.responses.create(input=prompt, **settings)     # one request, one full response
    seconds = time.time() - start                                    # time until the answer is back
    if response.status != "completed":                               # not a successful, finished answer
        raise RuntimeError(f"Response not completed: {response.status}, {response.incomplete_details}")   # stop with the reason
    return {"text": response.output_text,                            # the whole answer
            "response": response,                                    # the full response object
            "seconds": seconds}                                      # how long the call took

## Step 4 - Function 2: streaming response

`get_streamed_response()` sends the same request with `stream=True`. Instead of one response, the API sends a series of events while the model is writing. I go through them one by one:
- `response.output_text.delta`: a new piece of text. I print it right away and add it to a list.
- `response.completed`: the response finished successfully. It also carries the final response object.
- `response.incomplete`, `response.failed` or `error`: something went wrong, so I raise an error.
- All other events (`response.created`, `response.output_text.done`, ...) carry no new text, so I only count them.

To avoid duplication, I build the text only from the delta events. The `response.output_text.done` event repeats the whole text at the end, so adding it as well would print and store the answer twice. If the stream stops without a `response.completed` event, the function also raises an error, because then the answer may be cut off.

In [6]:
from collections import Counter   # to count the event types

def get_streamed_response(prompt, show=True, **settings):   # function 2
    # Streaming call: print the text while it arrives and rebuild the full text from the deltas
    pieces = []                       # every delta, in the order it arrived
    arrival = []                      # seconds after the start when each delta arrived
    events = Counter()                # how many events of each type
    final = None                      # the final response object, set by response.completed
    start = time.time()               # start the timer

    with client.responses.create(input=prompt, stream=True, **settings) as stream:   # same request, but streamed
        for event in stream:                                             # events arrive while the model writes
            events[event.type] += 1                                      # count every event type
            if event.type == "response.output_text.delta":               # a new piece of text
                pieces.append(event.delta)                               # keep it (exactly once)
                arrival.append(time.time() - start)                      # remember when it arrived
                if show:                                                 # printing can be switched off
                    print(event.delta, end="", flush=True)               # show it right away
            elif event.type == "response.completed":                     # finished successfully
                final = event.response                                   # the full final response
            elif event.type in ("response.incomplete", "response.failed"):   # finished, but not successfully
                r = event.response                                       # the response with the reason
                raise RuntimeError(f"Stream ended with {event.type}: {r.incomplete_details or r.error}")   # stop with the reason
            elif event.type == "error":                                  # error event from the API
                raise RuntimeError(f"Stream error {event.code}: {event.message}")   # stop with the message
            # other events (created, in_progress, output_text.done, ...) carry no new text

    if final is None:                                                    # the stream stopped without response.completed
        raise RuntimeError("The stream ended without response.completed - the answer may be cut off")   # don't trust it
    return {"text": "".join(pieces),                                     # full text rebuilt from the deltas
            "response": final,                                           # the final response object
            "pieces": pieces,                                            # the deltas themselves
            "arrival": arrival,                                          # when each delta arrived
            "events": events,                                            # event counts by type
            "seconds": time.time() - start}                              # total time

## Step 5 - Run both functions

First the complete response. The cell shows nothing until the whole answer is back.

In [7]:
complete = get_complete_response(PROMPT, **SETTINGS)                    # function 1
print(complete["text"])                                                 # the whole answer at once
print(f"\n--- status: {complete['response'].status} | time: {complete['seconds']:.1f} s"   # print the status, the time and the token count
      f" | output tokens: {complete['response'].usage.output_tokens}")    # short summary

Streaming a response from a large language model means sending the model’s output to the user in small pieces as it is generated, instead of waiting for the entire answer to be finished first. In practice, the app receives tokens or short chunks of text one by one and displays them immediately, so the reply appears to “type itself out” in real time.

This makes a chat application feel faster because the user sees progress almost right away. Even if the full answer takes several seconds to complete, the delay before the first visible output is much shorter. That reduces the feeling of waiting and reassures the user that the system is working. Streaming also improves the conversational experience: users can start reading sooner, perceive the app as more responsive, and sometimes interrupt or react before the answer is complete. So while total generation time may not change much, perceived speed and usability improve significantly.

--- status: completed | time: 5.8 s | output tokens: 178

Now the same prompt and settings with streaming. The text appears piece by piece while the model writes it.

In [8]:
streamed = get_streamed_response(PROMPT, **SETTINGS)                    # function 2 (prints while it streams)
print(f"\n\n--- status: {streamed['response'].status} | time: {streamed['seconds']:.1f} s"   # print the status, the time and the token count
      f" | first text after: {streamed['arrival'][0]:.2f} s"   # when the first piece arrived
      f" | output tokens: {streamed['response'].usage.output_tokens}")    # short summary

Streaming a response from a large language model means sending the answer to the user in small pieces as the model generates it, instead of waiting for the full response to be completed first. In practice, the server delivers tokens, words, or short text chunks incrementally, so the user sees the message appear gradually, much like someone typing in real time.

This makes a chat application feel faster because the user gets immediate feedback. Even if the full answer takes several seconds to generate, the first part may arrive almost right away, reducing the sense of waiting. That improves perceived performance and makes the interaction feel more responsive and conversational. Streaming also helps users start reading sooner, and in some cases they may get the information they need before the response is finished. Overall, streaming does not always reduce total generation time, but it greatly improves the user experience by shortening the apparent delay.

--- status: completed | time: 4

## Step 6 - Check the streamed text

These checks show that the streaming was real and that the text was rebuilt correctly:
- the rebuilt text must be exactly the same as the text in the final response object (nothing lost, nothing duplicated)
- the deltas must arrive spread out over time, not all at once
- the event counts show which events the API sent

In [9]:
text = streamed["text"]                                  # text rebuilt from the deltas
final_text = streamed["response"].output_text            # text in the final response object

print("Delta events received       :", len(streamed["pieces"]))                    # how many pieces
print("Rebuilt text == final text  :", text == final_text)                         # True = nothing lost or duplicated
print("Length rebuilt / final      :", len(text), "/", len(final_text))            # same length
print("Status                      :", streamed["response"].status)                # completed = success
print(f"First / last delta arrived  : {streamed['arrival'][0]:.2f} s / {streamed['arrival'][-1]:.2f} s")   # spread over time
print("\nFirst 10 deltas:", streamed["pieces"][:10])                               # what the pieces look like

print("\nEvent types received:")                                                  # heading for the list
for name, count in streamed["events"].items():                                     # every event type, in order of first arrival
    print(f"  {name:38s} {count}")                                                 # name and count

Delta events received       : 173
Rebuilt text == final text  : True
Length rebuilt / final      : 967 / 967
Status                      : completed
First / last delta arrived  : 1.26 s / 4.60 s

First 10 deltas: ['Streaming', ' a', ' response', ' from', ' a', ' large', ' language', ' model', ' means', ' sending']

Event types received:
  response.created                       1
  response.in_progress                   1
  response.output_item.added             1
  response.content_part.added            1
  response.output_text.delta             173
  response.output_text.done              1
  response.content_part.done             1
  response.output_item.done              1
  response.completed                     1


## Step 7 - What happens when a response doesn't finish

To check that both functions recognize a response that didn't finish successfully, I run them again with `max_output_tokens=16` (the smallest value allowed), so the answer gets cut off. The status is then `incomplete` instead of `completed`, and the stream ends with a `response.incomplete` event instead of `response.completed`. Both functions should stop with an error instead of returning half an answer.

In [10]:
short_settings = {**SETTINGS, "max_output_tokens": 16}     # same settings, but the answer is cut off after 16 tokens

try:                                                         # this should fail
    get_complete_response(PROMPT, **short_settings)          # function 1 with the short limit
except RuntimeError as e:                                    # the function noticed it didn't complete
    print("Function 1:", e)                                  # show the reason

try:                                                         # this should fail too
    get_streamed_response(PROMPT, **short_settings)          # function 2: starts printing, then stops
except RuntimeError as e:                                    # the function noticed it didn't complete
    print("\nFunction 2:", e)                                # show the reason

Function 1: Response not completed: incomplete, IncompleteDetails(reason='max_output_tokens')
Streaming a response from a large language model means sending the answer to
Function 2: Stream ended with response.incomplete: IncompleteDetails(reason='max_output_tokens')


## Observations

1. Both functions worked with the same prompt and the same settings (gpt-5.4, temperature 0, no reasoning, up to 800 tokens). Both answers finished with status `completed`, with 178 and 177 output tokens.

2. The streaming was real. The first piece of text arrived after 1.26 seconds and the last one after 4.60 seconds, and the 173 delta events came in spread out over that time. With the complete response I saw nothing for 5.8 seconds and then the whole answer at once. So with streaming I could start reading about 4.5 seconds earlier. The total time was about the same (4.9 s vs 5.8 s). The difference is just normal variation between two calls. Streaming doesn't make the model write faster, it only shows the text earlier.

3. The text was rebuilt without duplication. Joining the 173 deltas gave 967 characters, exactly the same text as `output_text` in the final response object (the check printed `True`). The deltas are single words with the space in front (see the first 10 in Step 6: `' a'`, `' response'`, ...), so I join them with `""` and not with `" "`, otherwise the spaces would be doubled.

4. The event counts show the order of a streamed response: one `response.created`, `response.in_progress`, `output_item.added` and `content_part.added` at the start, 173 `response.output_text.delta` events, then one `output_text.done`, `content_part.done`, `output_item.done` and finally `response.completed`. The `output_text.done` event contains the whole text again. If I had added it to my text as well, the answer would have been doubled (1,934 characters instead of 967).

5. Both functions recognized when the answer didn't finish. With `max_output_tokens=16`, function 1 got status `incomplete` and raised an error with the reason `max_output_tokens`. Function 2 printed the beginning of the answer ("Streaming a response from a large language model means sending the answer to"), then received `response.incomplete` instead of `response.completed`, and also stopped with the reason. Neither function returned half an answer as if it was complete.

Overall: the complete call is simpler, but the user waits for the whole answer. The streaming call shows text after about a second, and by building the text only from the delta events and waiting for `response.completed`, I get exactly the same final text and know that it is complete.

## Appendix - Concepts used in this problem

### 1. Complete response vs streaming

| | Complete response | Streaming (`stream=True`) |
|---|---|---|
| What the API sends | one response object when the answer is finished | many small events while the answer is being written |
| When the user sees text | only after the whole answer is done | as soon as the first piece is written |
| How I get the text | `response.output_text` | join all `response.output_text.delta` pieces |
| How I know it finished | `response.status == "completed"` | a `response.completed` event arrived |
| Tokens and cost | the same | the same |

Streaming doesn't make the model write faster. It makes the app *feel* faster, because the user can start reading after the first piece arrives instead of waiting for the whole answer.

### 2. How streaming works

With `stream=True` the connection stays open and the server sends **events** one after another (server-sent events). The OpenAI library turns them into Python objects, and `for event in stream:` reads them as they come in. `with ... as stream:` makes sure the connection is closed at the end, even after an error.

### 3. The events in a streamed response

| Event | What it means | Used in my function | In my run |
|---|---|---|---|
| `response.created` | the request was accepted | counted only | 1 |
| `response.in_progress` | the model started working | counted only | 1 |
| `response.output_item.added` / `response.content_part.added` | a new message / text part starts | counted only | 1 / 1 |
| **`response.output_text.delta`** | **a new piece of text (`event.delta`)** | **printed and added to the text** | **173** |
| `response.output_text.done` | the text part is finished; `event.text` holds the **whole** text again | counted only (adding it would duplicate the text) | 1 |
| `response.content_part.done` / `response.output_item.done` | the part / message is finished | counted only | 1 / 1 |
| **`response.completed`** | **the response finished successfully; `event.response` is the final response** | **marks success** | **1** |
| `response.incomplete` | the response stopped early (e.g. `max_output_tokens` reached) | raises an error | in the 16-token test |
| `response.failed` / `error` | the response or the stream failed | raises an error | 0 |

### 4. Rebuilding the text without duplication

A delta is only the new piece of text, usually about one token (173 deltas for 177 tokens in my run). The full text is all the deltas joined in order with nothing in between: `"".join(pieces)`, because each delta already contains its own space. Duplication happens if the same text is added twice, for example by also adding `response.output_text.done` (which repeats the whole text) or the final `response.output_text`.

In my run: 173 deltas → 967 characters, exactly equal to the final `output_text`. Adding the `done` text as well would have given 1,934 characters.

### 5. Recognizing successful completion

A stream can also stop early: the token limit is reached, the request fails, or the connection drops. So "the loop ended" doesn't mean "the answer is complete". Only a `response.completed` event (status `completed`) means success. `response.incomplete` comes with `incomplete_details.reason`, and `response.failed` comes with `error`.

| Status / event | Meaning | In my run |
|---|---|---|
| `completed` / `response.completed` | the whole answer was written | both normal calls |
| `incomplete` / `response.incomplete` | stopped early; the reason is in `incomplete_details` | both calls with `max_output_tokens=16` (reason `max_output_tokens`) |
| `failed` / `response.failed` | the request failed; the reason is in `error` | didn't happen |

### 6. Real streaming vs simulated streaming

Simulated streaming means waiting for the whole answer and then printing it in small pieces. It looks similar, but the user still waits for the full answer before seeing anything, so it brings no benefit. In real streaming the deltas arrive spread out over time, and the first one arrives long before the answer is finished. In my run they arrived between 1.26 s and 4.60 s.

### 7. Time to first token

| Measure | Complete response | Streaming | In my run |
|---|---|---|---|
| Time until the user sees text | the whole call | time to the first delta | 5.8 s vs **1.26 s** |
| Time until the answer is complete | the whole call | about the same as without streaming | 5.8 s vs 4.9 s |